# care-nudge training on Kaggle (fallback for when the Colab GPU quota runs out)

Before running:
1. **Settings → Accelerator → GPU T4 x2.** Don't pick P100: torch>=2.8 no longer supports Pascal GPUs.
2. **Settings → Internet → On.** The first cell clones the repo and pip-installs packages.
3. Optional: **Add-ons → Secrets** → add `HF_TOKEN` (a Hugging Face write token) and attach it to this notebook. Without it, Cell 2 asks you to paste the token.

Everything in `/kaggle/working` shows up in the Output panel, where you can download it.

In [ ]:
# Cell 1: fresh copy of the code and the training stack
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # T4 x2: use one GPU so 4-bit training isn't split across both
%cd /kaggle/working
!rm -rf /kaggle/working/care-nudge
!git clone https://github.com/SachiK0thari/care-nudge.git
%cd /kaggle/working/care-nudge
!pip install -q -r requirements-dev.txt

In [ ]:
# Cell 2: log in to Hugging Face (from the HF_TOKEN Kaggle secret if attached, else paste a write token)
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
except Exception:
    login()

In [ ]:
!nvidia-smi
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# Cell 3: data + two baselines from the un-tuned base model: the same prompt, then the prompt plus worked examples.
!python scripts/build_dataset.py
!python scripts/evaluate.py --model Qwen/Qwen3-4B-Instruct-2507 --name base
!python scripts/evaluate.py --model Qwen/Qwen3-4B-Instruct-2507 --few-shot --name base_fewshot

In [ ]:
# Cell 4: fine-tune, then evaluate the adapter on the test split and on the probe set
!python scripts/train.py --push-to SachiK/carenudge-lora
!python scripts/evaluate.py --model Qwen/Qwen3-4B-Instruct-2507 --adapter outputs/carenudge-lora --name finetuned
!python scripts/evaluate.py --model Qwen/Qwen3-4B-Instruct-2507 --adapter outputs/carenudge-lora --test data/probes.jsonl --name finetuned_probes

In [ ]:
# Cell 5: merge the adapter and publish the model the Space will load
!python scripts/merge.py --push-to SachiK/carenudge-qwen3-4b

In [ ]:
# Cell 6: bundle the reports so you can download them from the Output panel and copy them into the repo
!cd /kaggle/working/care-nudge && zip -qr /kaggle/working/reports.zip reports
!ls -lh /kaggle/working/reports.zip